In [1]:
pip install mysql-connector-python

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.5 MB 7.3 MB/s eta 0:00:03
   --------- ------------------------------ 3.9/16.5 MB 14.1 MB/s eta 0:00:01
   ------------ --------------------------- 5.0/16.5 MB 10.3 MB/s eta 0:00:02
   ------------- -------------------------- 5.5/16.5 MB 8.4 MB/s eta 0:00:02
   ---------------- ----------------------- 6.8/16.5 MB 8.0 MB/s eta 0:00:02
   ----------------- ---------------------- 7.3/16.5 MB 6.8 MB/s eta 0:00:02
   ------------------ --------------------- 7.6/16.5 MB 6.0 MB/s eta 0:00:02
   ------------------- -------------------- 8.1/16.5 MB 5.2 MB/s eta 0:00:02
   -------------------- ------------------- 8.4/16.5 MB 4.8 MB/s eta 0:00:02
   -------------------- ------------------- 8.7/16.5 MB 4.6 MB/s eta 0:00:02
   ---------------------- ----------------- 9.2/16.5 MB 4.2 MB/s eta 0:00:02
   ---------------------- ----------------- 9.4/16.5 MB 4.0 MB/s eta 0:00:02
   -

In [6]:
import mysql.connector
try:
    con = mysql.connector.connect(
        host="127.0.0.1",          # Try this instead of localhost sometimes helps
        user="root",
        password="NewStrongPass123!",
        database="emp"
    )
    print("Connected successfully!")
    con.close()
except mysql.connector.Error as err:
    print(f"Error: {err}")

Error: 1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)


In [8]:
def check_employee(employee_id):
    # Query to select all rows from the employees table where id matches
    sql = 'SELECT * FROM employees WHERE id=%s'
    # Making cursor buffered to make rowcount method work properly
    cursor = con.cursor(buffered=True)
    data = (employee_id,)
    # Executing the SQL Query
    cursor.execute(sql, data)
    # Fetch the first row to check if employee exists
    employee = cursor.fetchone()
    # Closing the cursor
    cursor.close()
    # If employee is found, return True, else return False
    return employee is not None

In [9]:
def add_employee():
    Id = input("Enter Employee Id: ")

    # Checking if Employee with given Id already exists
    if check_employee(Id):
        print("Employee already exists. Please try again.")
        return
    
    else:
        Name = input("Enter Employee Name: ")
        Post = input("Enter Employee Post: ")
        Salary = input("Enter Employee Salary: ")

        # Inserting Employee details into the employees table
        sql = 'INSERT INTO employees (id, name, position, salary) VALUES (%s, %s, %s, %s)'
        data = (Id, Name, Post, Salary)
        cursor = con.cursor()

        try:
            # Executing the SQL Query
            cursor.execute(sql, data)

            # Committing the transaction
            con.commit()
            print("Employee Added Successfully")
        
        except mysql.connector.Error as err:
            print(f"Error: {err}")
            con.rollback()
        
        finally:
            # Closing the cursor
            cursor.close()

In [10]:
def remove_employee():
    Id = input("Enter Employee Id: ")

    # Checking if Employee with given Id exists
    if not check_employee(Id):
        print("Employee does not exist. Please try again.")
        return
    
    else:
        # Query to delete employee from the employees table
        sql = 'DELETE FROM employees WHERE id=%s'
        data = (Id,)
        cursor = con.cursor()

        try:
            # Executing the SQL Query
            cursor.execute(sql, data)

            # Committing the transaction
            con.commit()
            print("Employee Removed Successfully")
        
        except mysql.connector.Error as err:
            print(f"Error: {err}")
            con.rollback()
        
        finally:
            # Closing the cursor
            cursor.close()

In [11]:
def promote_employee():
    Id = input("Enter Employee's Id: ")

    # Checking if Employee with given Id exists
    if not check_employee(Id):
        print("Employee does not exist. Please try again.")
        return
    
    else:
        try:
            Amount = float(input("Enter increase in Salary: "))

            # Query to Fetch Salary of Employee with given Id
            sql_select = 'SELECT salary FROM employees WHERE id=%s'
            data = (Id,)
            cursor = con.cursor()

            # Executing the SQL Query
            cursor.execute(sql_select, data)

            # Fetching Salary of Employee with given Id
            current_salary = cursor.fetchone()[0]
            new_salary = current_salary + Amount

            # Query to Update Salary of Employee with given Id
            sql_update = 'UPDATE employees SET salary=%s WHERE id=%s'
            data_update = (new_salary, Id)

            # Executing the SQL Query to update salary
            cursor.execute(sql_update, data_update)

            # Committing the transaction
            con.commit()
            print("Employee Promoted Successfully")

        except (ValueError, mysql.connector.Error) as e:
            print(f"Error: {e}")
            con.rollback()

        finally:
            # Closing the cursor
            cursor.close()

In [12]:
def display_employees():
    try:
        # Query to select all rows from the employees table
        sql = 'SELECT * FROM employees'
        cursor = con.cursor()

        # Executing the SQL Query
        cursor.execute(sql)

        # Fetching all details of all the Employees
        employees = cursor.fetchall()
        for employee in employees:
            print("Employee Id : ", employee[0])
            print("Employee Name : ", employee[1])
            print("Employee Post : ", employee[2])
            print("Employee Salary : ", employee[3])
            print("------------------------------------")

    except mysql.connector.Error as err:
        print(f"Error: {err}")

    finally:
        # Closing the cursor
        cursor.close()

In [13]:
def menu():
    while True:
        print("\nWelcome to Employee Management Record")
        print("Press:")
        print("1 to Add Employee")
        print("2 to Remove Employee")
        print("3 to Promote Employee")
        print("4 to Display Employees")
        print("5 to Exit")
        
        # Taking choice from user
        ch = input("Enter your Choice: ")

        if ch == '1':
            add_employee()
        elif ch == '2':
            remove_employee()
        elif ch == '3':
            promote_employee()
        elif ch == '4':
            display_employees()
        elif ch == '5':
            print("Exiting the program. Goodbye!")
            break
        else:
            print("Invalid Choice! Please try again.")

In [15]:
import mysql.connector
from mysql.connector import Error

# ==================== Database Connection ====================
def get_connection():
    try:
        con = mysql.connector.connect(
            host="localhost",
            user="root",
            password="NewStrongPass123!",   # ← CHANGE THIS TO YOUR ACTUAL ROOT PASSWORD
            database="emp"
        )
        print("Database connected successfully!")
        return con
    except Error as e:
        print(f"Error connecting to MySQL: {e}")
        return None

# ==================== Helper Function ====================
def check_employee(con, employee_id):
    cursor = con.cursor()
    try:
        sql = "SELECT * FROM employees WHERE id = %s"
        cursor.execute(sql, (employee_id,))
        return cursor.rowcount > 0
    finally:
        cursor.close()

# ==================== Add Employee ====================
def add_employee(con):
    cursor = con.cursor()
    try:
        Id = input("Enter Employee Id: ").strip()
        if check_employee(con, Id):
            print("Employee already exists. Please try again.")
            return

        Name = input("Enter Employee Name: ").strip()
        Post = input("Enter Employee Post: ").strip()
        Salary = input("Enter Employee Salary: ").strip()

        # Basic validation
        if not Id or not Name or not Post:
            print("Id, Name, and Post cannot be empty!")
            return
        try:
            salary_value = float(Salary)
            if salary_value <= 0:
                print("Salary must be a positive number!")
                return
        except ValueError:
            print("Invalid salary format! Please enter a number.")
            return

        sql = """
        INSERT INTO employees (id, name, position, salary)
        VALUES (%s, %s, %s, %s)
        """
        data = (Id, Name, Post, salary_value)

        cursor.execute(sql, data)
        con.commit()
        print("Employee Added Successfully!")
    except Error as err:
        print(f"Database error: {err}")
        con.rollback()
    finally:
        cursor.close()

# ==================== Remove Employee ====================
def remove_employee(con):
    cursor = con.cursor()
    try:
        Id = input("Enter Employee Id: ").strip()
        if not check_employee(con, Id):
            print("Employee does not exist. Please try again.")
            return

        confirm = input(f"Are you sure you want to delete employee {Id}? (yes/no): ").strip().lower()
        if confirm != 'yes':
            print("Deletion cancelled.")
            return

        sql = "DELETE FROM employees WHERE id = %s"
        cursor.execute(sql, (Id,))
        con.commit()
        print("Employee Removed Successfully!")
    except Error as err:
        print(f"Database error: {err}")
        con.rollback()
    finally:
        cursor.close()

# ==================== Promote Employee (Salary Increase) ====================
def promote_employee(con):
    cursor = con.cursor()
    try:
        Id = input("Enter Employee's Id: ").strip()
        if not check_employee(con, Id):
            print("Employee does not exist. Please try again.")
            return

        try:
            Amount = float(input("Enter increase in Salary: ").strip())
            if Amount <= 0:
                print("Increase amount must be positive!")
                return
        except ValueError:
            print("Invalid amount! Please enter a number.")
            return

        cursor.execute("SELECT salary FROM employees WHERE id = %s", (Id,))
        current_salary = cursor.fetchone()[0]
        new_salary = current_salary + Amount

        sql = "UPDATE employees SET salary = %s WHERE id = %s"
        cursor.execute(sql, (new_salary, Id))
        con.commit()
        print(f"Employee Promoted Successfully! New Salary: {new_salary}")
    except Error as err:
        print(f"Database error: {err}")
        con.rollback()
    except TypeError:
        print("Error fetching current salary.")
    finally:
        cursor.close()

# ==================== Display All Employees ====================
def display_employees(con):
    cursor = con.cursor()
    try:
        cursor.execute("SELECT * FROM employees ORDER BY id")
        employees = cursor.fetchall()

        if not employees:
            print("No employees found in the database.")
            return

        print("\n{:<10} {:<25} {:<20} {:<12}".format("ID", "Name", "Position", "Salary"))
        print("-" * 70)
        for emp in employees:
            print("{:<10} {:<25} {:<20} {:<12}".format(emp[0], emp[1], emp[2], f"{emp[3]:.2f}"))
    except Error as err:
        print(f"Error: {err}")
    finally:
        cursor.close()

# ==================== Menu ====================
def menu():
    con = get_connection()
    if con is None:
        print("Cannot start program without database connection.")
        return

    try:
        while True:
            print("\n" + "="*40)
            print("   EMPLOYEE MANAGEMENT SYSTEM")
            print("="*40)
            print("1. Add Employee")
            print("2. Remove Employee")
            print("3. Promote Employee (Salary Increase)")
            print("4. Display All Employees")
            print("5. Exit")
            print("-"*40)

            choice = input("Enter your choice (1-5): ").strip()

            if choice == '1':
                add_employee(con)
            elif choice == '2':
                remove_employee(con)
            elif choice == '3':
                promote_employee(con)
            elif choice == '4':
                display_employees(con)
            elif choice == '5':
                print("\nThank you for using the Employee Management System!")
                print("Goodbye!")
                break
            else:
                print("Invalid choice! Please enter a number between 1 and 5.")
    finally:
        if con.is_connected():
            con.close()
            print("Database connection closed.")

# ==================== Main ====================
if __name__ == "__main__":
    menu()

Error connecting to MySQL: 1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)
Cannot start program without database connection.
